# BOLT by example

This notebook is an end-to-end tour of the **Books of Life Toolkit (BOLT)**: it
transforms *complex log data* (registry-style records spread across many rows and
tables) into a single plain-text **book of life** per person, ready for analysis
with large language models.

It walks through:

1. Connecting to and inspecting the database
2. Reading a recipe (the *what / who / how*)
3. Writing a single book of life
4. Adding **social context** (books within books)
5. Defining a recipe inline in Python
6. Generating books at scale and saving to JSONL
7. Reading the saved dataset back

**Prerequisites.** Run from the repository root with the package installed
(`pip install -e .`). No data is required up front: the first code cell builds
`dbs/db.duckdb` from synthetic data automatically if it is missing, so the whole
notebook runs without any access to real registry data.

In [ ]:
import os
import sys
import json
import subprocess

import duckdb

# Run this notebook from the repository root so relative paths
# (dbs/, recipes/, synth/) resolve correctly.
if not os.path.exists("main.py"):
    raise RuntimeError(
        "Please run this notebook from the BooksOfLifeToolkit repository root."
    )

from serialization.Recipe import Recipe
from serialization.registry import INSTANTIATORS
from serialization.BookofLifeGenerator import BookofLifeGenerator
from serialization.BookofLifeGeneratorBatch import BookofLifeGeneratorBatch
from utils.utils import get_unique_rinpersoons, basic_gen, save_to_jsonl_shard

DB_PATH = "dbs/db.duckdb"

# BOLT ships no data. dbs/db.duckdb is built locally from synthetic data.
# If it does not exist yet, build it now: synthetic data -> CSVs -> DuckDB.
if not os.path.exists(DB_PATH):
    print("No database found - building one from synthetic data...")
    subprocess.run([sys.executable, "synth/main.py"], check=True)
    subprocess.run(
        [
            sys.executable, "serialization/make_db.py",
            "--data_dir", "synth/data",
            "--yaml_file", "recipes/make_db",
            "--db_name", "db",
        ],
        check=True,
    )
    print("Done.")
else:
    print(f"Using existing database at {DB_PATH}.")

## 1. Connect to and inspect the database

BOLT reads from a DuckDB database whose tables are "complex log data" sources
(here: demographics and household spells). Notice how one person's life is
spread across many rows in `household_bus` — exactly the structure BOLT turns
into a single readable narrative.

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)

# What tables (data sources) are in the database?
tables = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]
print("Tables:", tables)
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n:,} rows")

# Peek at a few raw rows of the household log - the kind of "complex log data"
# BOLT turns into text. A single person's life is spread across many rows.
print("\nSample household_bus rows (raw registry-style records):")
conn.sql(
    "SELECT rinpersoon, HUISHOUDNR, TYPHH, PLHH, DATUMAANVANGHH, DATUMEINDEHH "
    "FROM household_bus LIMIT 5"
).show()

## 2. The recipe: *what*, *who*, *how*

A **recipe** is the configuration that drives a book. It specifies the **what**
(which data sources and features to include), the **who** (social context), and
the **how** (ordering and formatting). Let's look at the template recipe.

In [ ]:
# A recipe declares the WHAT (information sources + features), the WHO
# (social context), and the HOW (ordering + formatting).
with open("recipes/template.yaml") as f:
    print(f.read())

recipe = Recipe("recipes/template.yaml")
print("main_key:         ", recipe.main_key)
print("datasets (what):  ", recipe.dataset_names)
print("sorting (how):    ", recipe.sorting_keys)
print("generator (how):  ", recipe.paragraph_generator)
print("registered sources:", sorted(INSTANTIATORS))

## 3. Write a single Book of Life

`BookofLifeGenerator` reads the recipe, pulls the relevant rows for one person
from the database, turns each into a paragraph, orders them, and renders the
final plain-text book.

In [ ]:
# Pick one individual and write their Book of Life from the template recipe.
rinpersoons = get_unique_rinpersoons(DB_PATH)
person = rinpersoons[0]

book = BookofLifeGenerator(person, "recipes/template.yaml", duck_db_conn=conn).generate_book()
print(f"Book of Life for {person}\n" + "=" * 60)
print(book)

## 4. Social context: books within books

A key strength of BOLT is the **who**: capturing the linked lives around a person.
The `social_context` recipe writes short nested books for the partners and children
in each household spell, embedded inside the focal person's book.

In [ ]:
# The "who": include other people (partners, children) as nested
# "books within books". Find someone living with a partner and children.
focal = conn.execute(
    "SELECT rinpersoon FROM household_bus "
    "WHERE PLHH IN ('3','4') AND AANTALKINDHH NOT IN ('0','nan') LIMIT 1"
).fetchone()[0]

social_book = BookofLifeGenerator(
    focal, "recipes/social_context.yaml", duck_db_conn=conn
).generate_book()
print(f"Book of Life (with social context) for {focal}\n" + "=" * 60)
print(social_book)

## 5. Define a recipe inline

Recipes are just dictionaries, so you can build them in Python without a YAML
file. This makes it easy to experiment with different *what / who / how* choices.

In [ ]:
# Recipes can also be built in Python as a dict - handy for experimentation.
# Here: a minimal "Book 1"-style book with only sex and year of birth.
inline_recipe = {
    "main_key": "rinpersoon",
    "datasets": [
        {"name": "persoon_tab", "features": ["GBAGESLACHT", "GBAGEBOORTEJAAR"]},
    ],
    "formatting": {
        "sorting_keys": ["year"],
        "paragraph_generator": "get_paragraph_string_tabular",
    },
}

mini_book = BookofLifeGenerator(person, inline_recipe, duck_db_conn=conn).generate_book()
print(f"Minimal book for {person}\n" + "=" * 60)
print(mini_book)

## 6. Generate at scale and save to JSONL

For real use you generate books for many people at once. `BookofLifeGeneratorBatch`
instantiates paragraphs for the whole population in one pass; the books are then
written out and saved as a sharded JSONL file (`output/books.jsonl`).

In [ ]:
# Generate books at scale and save them as JSONL (one record per person).
# This is the format used for downstream LLM work. The batch generator
# instantiates paragraphs for everyone in one pass, then we write each book.
batch_generator = BookofLifeGeneratorBatch(rinpersoons, "recipes/template.yaml", DB_PATH, conn)
batch_generator.write_books()

data_buffer = []
for rinpersoon, paragraphs in batch_generator.rin_dicts.items():
    rid, book_content = basic_gen(
        rinpersoon=rinpersoon,
        recipe_yaml_path="recipes/template.yaml",
        paragraphs=paragraphs,
        conn=conn,
    )
    data_buffer.append({"rinpersoon": rid, "book_content": book_content})

# Start from a clean file so re-running the notebook does not append duplicates.
out_file = os.path.join("output", "books.jsonl")
if os.path.exists(out_file):
    os.remove(out_file)
out_path = save_to_jsonl_shard(data_buffer, "output", shard_index=None)
print(f"Wrote {len(data_buffer):,} books to {out_path}")

# A rough sense of book length (characters and whitespace-delimited words).
char_lengths = [len(r["book_content"]) for r in data_buffer]
word_lengths = [len(r["book_content"].split()) for r in data_buffer]
print(
    f"Avg length: {sum(char_lengths) // len(char_lengths):,} chars / "
    f"{sum(word_lengths) // len(word_lengths):,} words per book"
)

## 7. Inspect the saved dataset

Finally, read the JSONL back and look at one record the way a downstream
consumer (e.g. an LLM fine-tuning or prompting pipeline) would.

In [ ]:
# Read the saved dataset back and show one record as it would be consumed downstream.
with open(os.path.join("output", "books.jsonl")) as f:
    first = json.loads(f.readline())

print("Keys:", list(first.keys()))
print("rinpersoon:", first["rinpersoon"])
print("\nbook_content (first 600 chars):\n")
print(first["book_content"][:600])